# MORPHEUS — Feature Engineering and Finalist Rerun

## 1. Notebook scope and rationale

This notebook introduces the first dedicated feature engineering stage of the MORPHEUS project.

The previous modeling notebook established a benchmark over the post-EDA base feature space and identified HistGradientBoostingClassifier, XGBoost, and CatBoost as the strongest provisional finalists. However, that benchmark was performed before any explicit feature engineering phase, which means that the finalist ranking reflects model behavior under the original analytical representation of the problem.

The purpose of this notebook is therefore to improve the feature space in a disciplined way before moving into final model refinement. The focus is not on arbitrary feature proliferation, but on constructing a compact set of analytically defensible derived variables from the existing active predictors, especially in the domains of blood pressure, adiposity, sleep-related symptoms, and socioeconomic context.

This notebook does not yet perform final hyperparameter tuning or threshold analysis. Instead, it prepares the project for those later stages by testing whether a more informative feature representation improves the performance of the current finalist models.

## 2. Inputs from the previous modeling stage

The previous benchmark identified three provisional finalists under the base feature space:

- HistGradientBoostingClassifier
- XGBoost
- CatBoost

These finalists are carried forward into the current notebook because they provided the strongest overall balance between discrimination, positive-class recovery, and probability quality in the earlier comparison.

At this stage, the goal is not to expand the model zoo further, but to test whether these leading candidates benefit from a more expressive and better structured feature representation.

## 3. Feature engineering strategy

The feature engineering strategy in this notebook is intentionally conservative and structured.

In this first iteration, only derived variables that can be constructed from the currently active modeling feature space are considered. Previously dropped variables are not reintroduced by default. A dropped variable may only be reconsidered in a later iteration if it is needed to construct a clearly defensible derived feature whose analytical value outweighs the original reason for exclusion.

This design choice keeps the workflow interpretable and avoids mixing two separate operations at once: improving the feature representation of the active dataset, and reopening variables that were already excluded during earlier stages for methodological reasons.

The current notebook therefore prioritizes derived features related to:
- blood pressure summaries and categories
- adiposity and central obesity
- sleep symptom burden
- grouped weekend sleep representation
- socioeconomic simplification through poverty and education groupings
- age grouping for interpretable non-linearity

## 4. Imports and setup

This section loads the libraries and reusable project utilities required for feature engineering, finalist reruns, and preprocessing recipe comparison.

### 4.1 Load libraries and project utilities

In [1]:
# Standard library
from pathlib import Path
import sys

# Third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

from sklearn.model_selection import cross_validate

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Make project src importable from the notebook
PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# MARS utilities
from src.utils.config import load_and_validate_config
from src.data.io import load_dataset
from src.data.split import split_features_target, make_train_test_split
from src.preprocessing.typing import get_target_column, get_feature_groups
from src.preprocessing.cleaning import drop_columns
from src.preprocessing.pipeline_builder import build_preprocessor

## 5. Data loading and baseline specification

This section reloads the project configuration and the assembled dataset, reconstructs the current modeling feature space, and restricts the workflow to the modelable cohort.

The purpose is to start from the same analytical baseline established in the previous modeling notebook before introducing any engineered variables.

### 5.1 Load configuration and assembled dataset

In [2]:
config = load_and_validate_config(PROJECT_ROOT / "configs" / "config.yaml")

data_path = PROJECT_ROOT / config["data"]["input_path"]
df = load_dataset(data_path, file_type=config["data"]["file_type"])

print("Config loaded successfully.")
print(f"Resolved data path: {data_path}")
print(f"Dataset shape: {df.shape}")

Config loaded successfully.
Resolved data path: F:\DOCUMENTOS\CIENCIA DE DATOS\portfolio-data-science\morpheus-sleep-health-ml\data\processed\nhanes_sleep_master_v1.csv
Dataset shape: (10195, 47)


### 5.2 Recover target and declared feature groups

In [3]:
target_col = get_target_column(config)
feature_groups = get_feature_groups(df, config)

numeric_features = feature_groups["numeric_features"]
categorical_features = feature_groups["categorical_features"]
drop_cols = config.get("drop_columns", [])

print(f"Target: {target_col}")
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Declared drop columns ({len(drop_cols)}): {drop_cols}")

Target: short_sleep
Numeric features (10): ['sleep_hours_weekend', 'age_years', 'income_to_poverty_ratio', 'weight_kg', 'height_cm', 'body_mass_index', 'waist_circumference_cm', 'sedentary_minutes', 'systolic_bp_mean', 'diastolic_bp_mean']
Categorical features (11): ['trouble_sleeping', 'told_doctor_sleep_disorder', 'snoring_frequency', 'sex', 'race_ethnicity', 'education_level', 'smoked_100_cigarettes_lifetime', 'alcohol_lifetime_12_drinks', 'alcohol_frequency', 'vigorous_activity', 'moderate_activity']
Declared drop columns (25): ['participant_id', 'sleep_hours_weekday', 'told_doctor_trouble_sleeping', 'systolic_bp_1', 'diastolic_bp_1', 'systolic_bp_2', 'diastolic_bp_2', 'systolic_bp_3', 'diastolic_bp_3', 'current_smoking_frequency', 'alcohol_past_year_12_drinks', 'drinks_per_day', 'vigorous_activity_days', 'vigorous_activity_minutes', 'moderate_activity_days', 'moderate_activity_minutes', 'phq_little_interest', 'phq_feeling_down', 'phq_sleep_problems', 'phq_low_energy', 'phq_appetit

### 5.3 Apply declared exclusions

In [4]:
df_model = drop_columns(df, drop_cols)

print(f"Original dataset shape: {df.shape}")
print(f"Post-drop dataset shape: {df_model.shape}")

dropped_now = [col for col in drop_cols if col not in df_model.columns]
print(f"Columns removed ({len(dropped_now)}): {dropped_now}")

Original dataset shape: (10195, 47)
Post-drop dataset shape: (10195, 22)
Columns removed (25): ['participant_id', 'sleep_hours_weekday', 'told_doctor_trouble_sleeping', 'systolic_bp_1', 'diastolic_bp_1', 'systolic_bp_2', 'diastolic_bp_2', 'systolic_bp_3', 'diastolic_bp_3', 'current_smoking_frequency', 'alcohol_past_year_12_drinks', 'drinks_per_day', 'vigorous_activity_days', 'vigorous_activity_minutes', 'moderate_activity_days', 'moderate_activity_minutes', 'phq_little_interest', 'phq_feeling_down', 'phq_sleep_problems', 'phq_low_energy', 'phq_appetite_problems', 'phq_feeling_bad_about_self', 'phq_trouble_concentrating', 'phq_slow_or_restless', 'phq_self_harm_thoughts']


### 5.4 Restrict to the modelable cohort

In [5]:
df_modelable = df_model[df_model[target_col].notna()].copy()

print(f"Post-drop dataset shape: {df_model.shape}")
print(f"Modelable cohort shape: {df_modelable.shape}")
print(f"Rows excluded due to missing target: {df_model.shape[0] - df_modelable.shape[0]}")

Post-drop dataset shape: (10195, 22)
Modelable cohort shape: (7288, 22)
Rows excluded due to missing target: 2907


### 5.5 Verify the baseline modeling specification

In [6]:
expected_columns = {target_col, *numeric_features, *categorical_features}
actual_columns = set(df_modelable.columns)

missing_expected = sorted(expected_columns - actual_columns)
unexpected_kept = sorted(actual_columns - expected_columns)

print(f"Expected modeling columns (including target): {len(expected_columns)}")
print(f"Actual columns in df_modelable: {len(actual_columns)}")

if missing_expected:
    print(f"Missing expected columns: {missing_expected}")
else:
    print("No expected columns are missing in df_modelable.")

if unexpected_kept:
    print(f"Columns still present but not declared in target/features: {unexpected_kept}")
else:
    print("No undeclared columns remain in df_modelable.")

Expected modeling columns (including target): 22
Actual columns in df_modelable: 22
No expected columns are missing in df_modelable.
No undeclared columns remain in df_modelable.


## 6. Base feature space before engineering

This section restates the active feature space that enters the feature engineering stage.

The objective is to make explicit which variables are currently available for derived feature construction before any transformation is introduced. This also makes the later comparison between the original and engineered feature spaces easier to interpret.

### 6.1 Summarize the active baseline feature space

In [7]:
base_feature_space_summary = pd.DataFrame(
    {
        "feature_group": ["numeric", "categorical", "total"],
        "n_features": [
            len(numeric_features),
            len(categorical_features),
            len(numeric_features) + len(categorical_features),
        ],
    }
)

base_feature_space_summary

,feature_group,n_features
0,numeric,10
1,categorical,11
2,total,21


### 6.2 Display the active baseline feature inventory

In [8]:
baseline_feature_inventory = pd.DataFrame(
    {
        "feature": numeric_features + categorical_features,
        "feature_group": (["numeric"] * len(numeric_features)) + (["categorical"] * len(categorical_features)),
    }
)

baseline_feature_inventory

,feature,feature_group
0,sleep_hours_weekend,numeric
1,age_years,numeric
2,income_to_poverty_ratio,numeric
3,weight_kg,numeric
4,height_cm,numeric
5,body_mass_index,numeric
6,waist_circumference_cm,numeric
7,sedentary_minutes,numeric
8,systolic_bp_mean,numeric
9,diastolic_bp_mean,numeric


### 6.3 Separate the baseline modeling matrix and target vector

In [9]:
X_base, y_base = split_features_target(df_modelable, target_col)

print(f"X_base shape: {X_base.shape}")
print(f"y_base shape: {y_base.shape}")
print(f"Target name: {y_base.name}")

X_base shape: (7288, 21)
y_base shape: (7288,)
Target name: short_sleep


## 7. Candidate engineered features

The candidate engineered features introduced in this notebook are selected on the basis of analytical plausibility, interpretability, and support from common epidemiological or clinical practice.

The aim is not to generate as many variables as possible, but to construct a compact set of derived features that may capture structure not fully represented by the raw variables alone. In particular, the current engineering stage focuses on:
- hemodynamic summaries and clinically interpretable blood pressure categories
- adiposity indicators beyond BMI alone
- sleep-related symptom proxies
- grouped weekend sleep representation
- socioeconomic and demographic simplifications that may support interpretable non-linearity

### 7.2 Feature-family rationale

The selected engineered features fall into five main families:

- **Blood pressure-derived features**, intended to summarize hemodynamic burden in a more interpretable way than systolic and diastolic pressure alone.
- **Adiposity-derived features**, intended to capture central obesity and clinically meaningful body-size categories beyond raw anthropometric inputs.
- **Sleep-related proxy features**, intended to summarize snoring, sleep-disorder indication, and symptom burden without introducing direct target leakage.
- **Weekend sleep grouping**, intended to retain interpretable information from weekend sleep duration without relying on weekday sleep, which defines the target.
- **Socioeconomic and demographic regroupings**, intended to support interpretable non-linearity and simplify common social gradients linked to sleep health.

## 8. Construction of engineered features

This section constructs the first engineered feature set from the active modeling variables.

All derived variables created here are designed to remain interpretable and analytically defensible. Features that are too speculative, too weakly supported, or too close to the target definition are intentionally excluded from this first iteration.

### 8.1 Blood pressure-derived features

Blood pressure is already present in the base feature space through mean systolic and diastolic values. In this section, those raw measurements are translated into more interpretable hemodynamic summaries and a clinically meaningful blood pressure category.

The goal is not to replace the original information blindly, but to test whether derived cardiovascular representations provide additional structure that may be useful for short-sleep prediction.

### 8.1.1 Create blood pressure-derived features

In [10]:
df_fe = df_modelable.copy()

In [11]:
df_fe = df_modelable.copy()

df_fe["pulse_pressure"] = (
    df_fe["systolic_bp_mean"] - df_fe["diastolic_bp_mean"]
)

df_fe["mean_arterial_pressure"] = (
    df_fe["diastolic_bp_mean"] + (df_fe["pulse_pressure"] / 3)
)

bp_missing = df_fe["systolic_bp_mean"].isna() | df_fe["diastolic_bp_mean"].isna()

bp_category = np.select(
    [
        (df_fe["systolic_bp_mean"] < 120) & (df_fe["diastolic_bp_mean"] < 80),
        (df_fe["systolic_bp_mean"].between(120, 129, inclusive="both")) & (df_fe["diastolic_bp_mean"] < 80),
        (df_fe["systolic_bp_mean"].between(130, 139, inclusive="both"))
        | (df_fe["diastolic_bp_mean"].between(80, 89, inclusive="both")),
        (df_fe["systolic_bp_mean"] >= 140) | (df_fe["diastolic_bp_mean"] >= 90),
    ],
    [
        "normal",
        "elevated",
        "stage_1",
        "stage_2",
    ],
    default=None,
)

df_fe["bp_category_accaha2017"] = pd.Series(bp_category, index=df_fe.index)
df_fe.loc[bp_missing, "bp_category_accaha2017"] = np.nan

df_fe[
    [
        "systolic_bp_mean",
        "diastolic_bp_mean",
        "pulse_pressure",
        "mean_arterial_pressure",
        "bp_category_accaha2017",
    ]
].head()

,systolic_bp_mean,diastolic_bp_mean,pulse_pressure,mean_arterial_pressure,bp_category_accaha2017
0,NaN,54.333333,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,107.000000,67.000000,40.000000,80.333333,normal
4,113.666667,67.333333,46.333333,82.777778,normal


### 8.1.2 Quick structural review of blood pressure-derived features

In [12]:
bp_feature_review = pd.DataFrame(
    {
        "missing_pct": df_fe[["pulse_pressure", "mean_arterial_pressure", "bp_category_accaha2017"]]
        .isna()
        .mean()
        .mul(100)
        .round(2),
        "n_unique_non_null": df_fe[["pulse_pressure", "mean_arterial_pressure", "bp_category_accaha2017"]]
        .nunique(dropna=True),
    }
)

bp_feature_review

,missing_pct,n_unique_non_null
pulse_pressure,16.74,1051
mean_arterial_pressure,16.74,1144
bp_category_accaha2017,17.71,4


In [13]:
df_fe["bp_category_accaha2017"].value_counts(dropna=False)

bp_category_accaha2017
normal      2738
stage_1     1637
NaN         1291
stage_2      820
elevated     802
Name: count, dtype: int64

### 8.2 Adiposity-derived features

Anthropometric variables in the base feature space already include weight, height, BMI, and waist circumference. This block introduces derived features intended to represent central adiposity and clinically interpretable body-size groupings more explicitly.

The rationale is that short sleep has been repeatedly linked to obesity and, in particular, to abdominal or central adiposity in epidemiological research. These derived features aim to test whether that structure is better captured through interpretable transformations than through the raw variables alone.

### 8.2.1 Create adiposity-derived features

In [14]:
df_fe["waist_to_height_ratio"] = (
    df_fe["waist_circumference_cm"] / df_fe["height_cm"]
)

df_fe["central_obesity_whtR_flag"] = np.where(
    df_fe["waist_to_height_ratio"].notna(),
    (df_fe["waist_to_height_ratio"] >= 0.5).astype(int),
    np.nan,
)

df_fe["bmi_category"] = pd.cut(
    df_fe["body_mass_index"],
    bins=[-np.inf, 18.5, 25, 30, 35, 40, np.inf],
    labels=[
        "underweight",
        "normal_weight",
        "overweight",
        "obesity_class_1",
        "obesity_class_2",
        "obesity_class_3",
    ],
    right=False,
)

df_fe[
    [
        "body_mass_index",
        "waist_circumference_cm",
        "height_cm",
        "waist_to_height_ratio",
        "central_obesity_whtR_flag",
        "bmi_category",
    ]
].head()

,body_mass_index,waist_circumference_cm,height_cm,waist_to_height_ratio,central_obesity_whtR_flag,bmi_category
0,37.8,117.9,160.2,0.735955,1.0,obesity_class_2
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,29.7,120.4,182.3,0.660450,1.0,overweight
4,21.9,86.8,184.2,0.471227,0.0,normal_weight


### 8.2.2 Quick structural review of adiposity-derived features

These derived features should remain simple and interpretable, but they still need a quick structural check before being carried forward into the updated feature space.

In [15]:
adiposity_feature_review = pd.DataFrame(
    {
        "missing_pct": df_fe[
            ["waist_to_height_ratio", "central_obesity_whtR_flag", "bmi_category"]
        ]
        .isna()
        .mean()
        .mul(100)
        .round(2),
        "n_unique_non_null": df_fe[
            ["waist_to_height_ratio", "central_obesity_whtR_flag", "bmi_category"]
        ].nunique(dropna=True),
    }
)

adiposity_feature_review

,missing_pct,n_unique_non_null
waist_to_height_ratio,12.50,6249
central_obesity_whtR_flag,12.50,2
bmi_category,8.97,6


### 8.2.4 Inspect categorical distributions

In [16]:
print("central_obesity_whtR_flag")
display(df_fe["central_obesity_whtR_flag"].value_counts(dropna=False))

print("\nbmi_category")
display(df_fe["bmi_category"].value_counts(dropna=False))

central_obesity_whtR_flag


central_obesity_whtR_flag
1.0    5147
0.0    1230
NaN     911
Name: count, dtype: int64


bmi_category


bmi_category
overweight         2040
normal_weight      1721
obesity_class_1    1380
obesity_class_2     738
NaN                 654
obesity_class_3     617
underweight         138
Name: count, dtype: int64

### 8.3 Sleep-related proxy features

This block introduces a small set of derived sleep-related proxy features built from the active symptom variables.

The objective is not to recreate a formal clinical screening instrument, but to summarize interpretable signals related to snoring, sleep-disorder indication, and overall sleep symptom burden without introducing direct target leakage.

### 8.3.1 Create sleep-related proxy features

In [17]:
df_fe["frequent_snoring_flag"] = np.where(
    df_fe["snoring_frequency"].isna(),
    np.nan,
    df_fe["snoring_frequency"].isin([3.0, 4.0]).astype(int),
)

df_fe["told_doctor_sleep_disorder_flag"] = np.where(
    df_fe["told_doctor_sleep_disorder"].isna(),
    np.nan,
    (df_fe["told_doctor_sleep_disorder"] == 1.0).astype(int),
)

df_fe["trouble_sleeping_flag"] = np.where(
    df_fe["trouble_sleeping"].isna(),
    np.nan,
    df_fe["trouble_sleeping"].isin([1.0, 2.0]).astype(int),
)

df_fe["possible_sdb_risk"] = np.where(
    df_fe[["frequent_snoring_flag", "told_doctor_sleep_disorder_flag"]].isna().all(axis=1),
    np.nan,
    (
        (df_fe["frequent_snoring_flag"] == 1)
        | (df_fe["told_doctor_sleep_disorder_flag"] == 1)
    ).astype(int),
)

df_fe["sleep_symptom_burden"] = (
    df_fe["trouble_sleeping_flag"].fillna(0)
    + df_fe["told_doctor_sleep_disorder_flag"].fillna(0)
    + df_fe["frequent_snoring_flag"].fillna(0)
)

all_missing_sleep_inputs = df_fe[
    ["trouble_sleeping_flag", "told_doctor_sleep_disorder_flag", "frequent_snoring_flag"]
].isna().all(axis=1)

df_fe.loc[all_missing_sleep_inputs, "sleep_symptom_burden"] = np.nan

df_fe[
    [
        "trouble_sleeping",
        "trouble_sleeping_flag",
        "told_doctor_sleep_disorder",
        "told_doctor_sleep_disorder_flag",
        "snoring_frequency",
        "frequent_snoring_flag",
        "possible_sdb_risk",
        "sleep_symptom_burden",
    ]
].head()

,trouble_sleeping,trouble_sleeping_flag,told_doctor_sleep_disorder,told_doctor_sleep_disorder_flag,snoring_frequency,frequent_snoring_flag,possible_sdb_risk,sleep_symptom_burden
0,1.0,1.0,2.0,0.0,NaN,NaN,0.0,1.0
1,NaN,NaN,2.0,0.0,2.0,0.0,0.0,0.0
2,NaN,NaN,2.0,0.0,1.0,0.0,0.0,0.0
3,NaN,NaN,1.0,1.0,3.0,1.0,1.0,2.0
4,NaN,NaN,1.0,1.0,2.0,0.0,1.0,1.0


### 8.3.2 Quick structural review of sleep-related proxy features

In [18]:
sleep_proxy_review = pd.DataFrame(
    {
        "missing_pct": df_fe[
            ["frequent_snoring_flag", "possible_sdb_risk", "sleep_symptom_burden"]
        ]
        .isna()
        .mean()
        .mul(100)
        .round(2),
        "n_unique_non_null": df_fe[
            ["frequent_snoring_flag", "possible_sdb_risk", "sleep_symptom_burden"]
        ].nunique(dropna=True),
    }
)

sleep_proxy_review

,missing_pct,n_unique_non_null
frequent_snoring_flag,17.00,2
possible_sdb_risk,0.03,2
sleep_symptom_burden,0.00,4


### 8.3.3 Inspect sleep-related categorical distributions

In [19]:
print("frequent_snoring_flag")
display(df_fe["frequent_snoring_flag"].value_counts(dropna=False))

print("\npossible_sdb_risk")
display(df_fe["possible_sdb_risk"].value_counts(dropna=False))

print("\nsleep_symptom_burden")
display(df_fe["sleep_symptom_burden"].value_counts(dropna=False).sort_index())

frequent_snoring_flag


frequent_snoring_flag
0.0    4092
1.0    1957
NaN    1239
Name: count, dtype: int64


possible_sdb_risk


possible_sdb_risk
0.0    4199
1.0    3087
NaN       2
Name: count, dtype: int64


sleep_symptom_burden


sleep_symptom_burden
0.0    2433
1.0    3122
2.0    1440
3.0     293
Name: count, dtype: int64

### 8.3 Sleep-related symptom features

In [20]:
print("frequent_snoring_flag")
display(df_fe["frequent_snoring_flag"].value_counts(dropna=False))

print("\npossible_sdb_risk")
display(df_fe["possible_sdb_risk"].value_counts(dropna=False))

print("\nsleep_symptom_burden")
display(df_fe["sleep_symptom_burden"].value_counts(dropna=False).sort_index())

frequent_snoring_flag


frequent_snoring_flag
0.0    4092
1.0    1957
NaN    1239
Name: count, dtype: int64


possible_sdb_risk


possible_sdb_risk
0.0    4199
1.0    3087
NaN       2
Name: count, dtype: int64


sleep_symptom_burden


sleep_symptom_burden
0.0    2433
1.0    3122
2.0    1440
3.0     293
Name: count, dtype: int64

### 8.4 Socioeconomic and grouped contextual features

This block introduces a small set of grouped contextual features intended to simplify socioeconomic and demographic structure without overcomplicating the feature space.

The goal is to retain interpretable signals related to social disadvantage, age-related heterogeneity, education grouping, and weekend sleep behavior, while keeping the resulting variables easy to explain within the broader sleep-health narrative of the project.

### 8.4.1 Create socioeconomic and grouped contextual features

In [21]:
df_fe["poverty_flag"] = np.where(
    df_fe["income_to_poverty_ratio"].isna(),
    np.nan,
    (df_fe["income_to_poverty_ratio"] < 1.0).astype(int),
)

df_fe["weekend_sleep_category"] = pd.cut(
    df_fe["sleep_hours_weekend"],
    bins=[-np.inf, 7, 9, np.inf],
    labels=["short_weekend_sleep", "recommended_weekend_sleep", "long_weekend_sleep"],
    right=False,
)

df_fe["age_band"] = pd.cut(
    df_fe["age_years"],
    bins=[18, 40, 60, np.inf],
    labels=["18_39", "40_59", "60_plus"],
    right=False,
)

education_map = {
    1.0: "low",
    2.0: "low",
    3.0: "medium",
    4.0: "medium",
    5.0: "high",
}

df_fe["education_group"] = df_fe["education_level"].replace(education_map)

df_fe[
    [
        "income_to_poverty_ratio",
        "poverty_flag",
        "sleep_hours_weekend",
        "weekend_sleep_category",
        "age_years",
        "age_band",
        "education_level",
        "education_group",
    ]
].head()

,income_to_poverty_ratio,poverty_flag,sleep_hours_weekend,weekend_sleep_category,age_years,age_band,education_level,education_group
0,5.00,0.0,8.0,recommended_weekend_sleep,29.0,18_39,5.0,high
1,5.00,0.0,8.0,recommended_weekend_sleep,21.0,18_39,4.0,medium
2,1.66,0.0,8.0,recommended_weekend_sleep,18.0,18_39,NaN,NaN
3,NaN,NaN,13.0,long_weekend_sleep,49.0,40_59,2.0,low
4,0.83,1.0,8.0,recommended_weekend_sleep,36.0,18_39,4.0,medium


### 8.4.2 Quick structural review of grouped contextual features

In [22]:
context_feature_review = pd.DataFrame(
    {
        "missing_pct": df_fe[
            ["poverty_flag", "weekend_sleep_category", "age_band", "education_group"]
        ]
        .isna()
        .mean()
        .mul(100)
        .round(2),
        "n_unique_non_null": df_fe[
            ["poverty_flag", "weekend_sleep_category", "age_band", "education_group"]
        ].nunique(dropna=True),
    }
)

context_feature_review

,missing_pct,n_unique_non_null
poverty_flag,15.75,2
weekend_sleep_category,21.76,3
age_band,5.36,3
education_group,9.95,3


### 8.4.3 Inspect categorical distributions

In [23]:
for col in ["poverty_flag", "weekend_sleep_category", "age_band", "education_group"]:
    print(col)
    display(df_fe[col].value_counts(dropna=False))
    print()

poverty_flag


poverty_flag
0.0    4946
1.0    1194
NaN    1148
Name: count, dtype: int64


weekend_sleep_category


weekend_sleep_category
recommended_weekend_sleep    2391
long_weekend_sleep           2021
NaN                          1586
short_weekend_sleep          1290
Name: count, dtype: int64


age_band


age_band
60_plus    2439
18_39      2334
40_59      2124
NaN         391
Name: count, dtype: int64


education_group


education_group
medium    3718
high      1610
low       1235
NaN        725
Name: count, dtype: int64

## 9. Quick review of engineered feature quality

Before rerunning the finalist models, the engineered variables should be reviewed at a basic structural level.

The purpose of this review is not to perform a full new EDA, but to confirm that the newly created variables are technically coherent, interpretable, and not dominated by missingness, degenerate categories, or obvious construction errors.

### 9.1 Consolidated structural summary of engineered features

In [24]:
engineered_feature_names = [
    "pulse_pressure",
    "mean_arterial_pressure",
    "bp_category_accaha2017",
    "waist_to_height_ratio",
    "central_obesity_whtR_flag",
    "bmi_category",
    "frequent_snoring_flag",
    "possible_sdb_risk",
    "sleep_symptom_burden",
    "poverty_flag",
    "weekend_sleep_category",
    "age_band",
    "education_group",
]

engineered_feature_review = pd.DataFrame(
    {
        "dtype": df_fe[engineered_feature_names].dtypes.astype(str),
        "missing_pct": df_fe[engineered_feature_names].isna().mean().mul(100).round(2),
        "n_unique_non_null": df_fe[engineered_feature_names].nunique(dropna=True),
    }
).sort_values(["missing_pct", "n_unique_non_null"], ascending=[False, True])

engineered_feature_review

,dtype,missing_pct,n_unique_non_null
weekend_sleep_category,category,21.76,3
bp_category_accaha2017,str,17.71,4
frequent_snoring_flag,float64,17.00,2
pulse_pressure,float64,16.74,1051
mean_arterial_pressure,float64,16.74,1144
poverty_flag,float64,15.75,2
central_obesity_whtR_flag,float64,12.50,2
waist_to_height_ratio,float64,12.50,6249
education_group,object,9.95,3
bmi_category,category,8.97,6


### 9.2 First plausibility checks

In [25]:
engineered_feature_plausibility = {
    "pulse_pressure_min": df_fe["pulse_pressure"].min(),
    "pulse_pressure_max": df_fe["pulse_pressure"].max(),
    "mean_arterial_pressure_min": df_fe["mean_arterial_pressure"].min(),
    "mean_arterial_pressure_max": df_fe["mean_arterial_pressure"].max(),
    "waist_to_height_ratio_min": df_fe["waist_to_height_ratio"].min(),
    "waist_to_height_ratio_max": df_fe["waist_to_height_ratio"].max(),
    "sleep_symptom_burden_min": df_fe["sleep_symptom_burden"].min(),
    "sleep_symptom_burden_max": df_fe["sleep_symptom_burden"].max(),
}

pd.Series(engineered_feature_plausibility)

pulse_pressure_min             20.000000
pulse_pressure_max            127.000000
mean_arterial_pressure_min     54.888889
mean_arterial_pressure_max    163.111111
waist_to_height_ratio_min       0.362431
waist_to_height_ratio_max       1.126126
sleep_symptom_burden_min        0.000000
sleep_symptom_burden_max        3.000000
dtype: float64

### 9.3 Inspect compact distributions of grouped engineered features

In [26]:
for col in [
    "bp_category_accaha2017",
    "bmi_category",
    "frequent_snoring_flag",
    "possible_sdb_risk",
    "poverty_flag",
    "weekend_sleep_category",
    "age_band",
    "education_group",
]:
    print(col)
    display(df_fe[col].value_counts(dropna=False))
    print()

bp_category_accaha2017


bp_category_accaha2017
normal      2738
stage_1     1637
NaN         1291
stage_2      820
elevated     802
Name: count, dtype: int64


bmi_category


bmi_category
overweight         2040
normal_weight      1721
obesity_class_1    1380
obesity_class_2     738
NaN                 654
obesity_class_3     617
underweight         138
Name: count, dtype: int64


frequent_snoring_flag


...

weekend_sleep_category
recommended_weekend_sleep    2391
long_weekend_sleep           2021
NaN                          1586
short_weekend_sleep          1290
Name: count, dtype: int64


age_band


age_band
60_plus    2439
18_39      2334
40_59      2124
NaN         391
Name: count, dtype: int64


education_group


education_group
medium    3718
high      1610
low       1235
NaN        725
Name: count, dtype: int64

In [27]:
display(df_fe["snoring_frequency"].value_counts(dropna=False))

snoring_frequency
2.0    2411
1.0    1681
3.0    1303
NaN    1239
4.0     654
Name: count, dtype: int64

In [28]:
display(df_fe["education_level"].value_counts(dropna=False).sort_index())

education_level
1.0     472
2.0     763
3.0    1597
4.0    2121
5.0    1610
NaN     725
Name: count, dtype: int64

In [29]:
display(df_fe["trouble_sleeping"].value_counts(dropna=False).sort_index())
display(df_fe["told_doctor_sleep_disorder"].value_counts(dropna=False).sort_index())
display(df_fe["sleep_symptom_burden"].value_counts(dropna=False).sort_index())

trouble_sleeping
1.0    1724
2.0    1231
3.0    1836
NaN    2497
Name: count, dtype: int64

told_doctor_sleep_disorder
1.0    1969
2.0    5313
NaN       6
Name: count, dtype: int64

sleep_symptom_burden
0.0    2433
1.0    3122
2.0    1440
3.0     293
Name: count, dtype: int64

## 10. Updated modeling feature space

This section defines the updated modeling specification after feature engineering.

At this point, the project moves from the base feature space used in the previous benchmark to an expanded but still disciplined feature representation. The objective is to test whether these additions improve the performance of the finalist models without sacrificing interpretability or methodological coherence.

### 10.1 Define the engineered feature set to carry forward

In [30]:
engineered_numeric_features = [
    "pulse_pressure",
    "mean_arterial_pressure",
    "waist_to_height_ratio",
    "sleep_symptom_burden",
]

engineered_categorical_features = [
    "bp_category_accaha2017",
    "central_obesity_whtR_flag",
    "bmi_category",
    "frequent_snoring_flag",
    "possible_sdb_risk",
    "poverty_flag",
    "weekend_sleep_category",
    "age_band",
    "education_group",
]

print("Engineered numeric features:")
print(engineered_numeric_features)

print("\nEngineered categorical features:")
print(engineered_categorical_features)

Engineered numeric features:
['pulse_pressure', 'mean_arterial_pressure', 'waist_to_height_ratio', 'sleep_symptom_burden']

Engineered categorical features:
['bp_category_accaha2017', 'central_obesity_whtR_flag', 'bmi_category', 'frequent_snoring_flag', 'possible_sdb_risk', 'poverty_flag', 'weekend_sleep_category', 'age_band', 'education_group']


### 10.2 Build the updated feature lists

In [31]:
numeric_features_fe = numeric_features + engineered_numeric_features
categorical_features_fe = categorical_features + engineered_categorical_features

print(f"Base numeric features: {len(numeric_features)}")
print(f"Engineered numeric features: {len(engineered_numeric_features)}")
print(f"Updated numeric features: {len(numeric_features_fe)}")

print(f"\nBase categorical features: {len(categorical_features)}")
print(f"Engineered categorical features: {len(engineered_categorical_features)}")
print(f"Updated categorical features: {len(categorical_features_fe)}")

Base numeric features: 10
Engineered numeric features: 4
Updated numeric features: 14

Base categorical features: 11
Engineered categorical features: 9
Updated categorical features: 20


### 10.3 Quick duplicate check in the updated feature space

In [32]:
duplicate_numeric = sorted(pd.Series(numeric_features_fe).value_counts().loc[lambda s: s > 1].index.tolist())
duplicate_categorical = sorted(pd.Series(categorical_features_fe).value_counts().loc[lambda s: s > 1].index.tolist())
cross_group_overlap = sorted(set(numeric_features_fe) & set(categorical_features_fe))

print(f"Duplicate numeric features: {duplicate_numeric}")
print(f"Duplicate categorical features: {duplicate_categorical}")
print(f"Cross-group overlap: {cross_group_overlap}")

Duplicate numeric features: []
Duplicate categorical features: []
Cross-group overlap: []


## 11. Rebuil train-test inputs for finalist rerun

This section rebuilds the modeling inputs after feature engineering so that the finalist models can be rerun on the updated representation of the problem.

The purpose is to preserve the same target definition and train-test logic used in the previous benchmark while allowing the engineered variables to enter the workflow in a controlled and explicit way.

### 11.1 Build the updated modeling matrix and target vector

In [33]:
X_fe = df_fe[numeric_features_fe + categorical_features_fe].copy()
y_fe = df_fe[target_col].copy()

print(f"X_fe shape: {X_fe.shape}")
print(f"y_fe shape: {y_fe.shape}")
print(f"Target name: {y_fe.name}")

X_fe shape: (7288, 34)
y_fe shape: (7288,)
Target name: short_sleep


### 11.2 Verify the updated feature matrix

In [34]:
expected_feature_columns_fe = numeric_features_fe + categorical_features_fe
actual_feature_columns_fe = X_fe.columns.tolist()

missing_in_X_fe = sorted(set(expected_feature_columns_fe) - set(actual_feature_columns_fe))
unexpected_in_X_fe = sorted(set(actual_feature_columns_fe) - set(expected_feature_columns_fe))

print(f"Declared updated feature columns: {len(expected_feature_columns_fe)}")
print(f"Actual columns in X_fe: {len(actual_feature_columns_fe)}")

if missing_in_X_fe:
    print(f"Missing columns in X_fe: {missing_in_X_fe}")
else:
    print("All updated feature columns are present in X_fe.")

if unexpected_in_X_fe:
    print(f"Unexpected columns in X_fe: {unexpected_in_X_fe}")
else:
    print("No undeclared columns are present in X_fe.")

Declared updated feature columns: 34
Actual columns in X_fe: 34
All updated feature columns are present in X_fe.
No undeclared columns are present in X_fe.


### 11.3 Create the updated train-test split

In [35]:
X_train_fe, X_test_fe, y_train_fe, y_test_fe = make_train_test_split(
    X_fe,
    y_fe,
    test_size=config["split"]["test_size"],
    random_state=config["split"]["random_state"],
    stratify=config["split"]["stratify"],
)

print(f"X_train_fe shape: {X_train_fe.shape}")
print(f"X_test_fe shape: {X_test_fe.shape}")
print(f"y_train_fe shape: {y_train_fe.shape}")
print(f"y_test_fe shape: {y_test_fe.shape}")

X_train_fe shape: (5830, 34)
X_test_fe shape: (1458, 34)
y_train_fe shape: (5830,)
y_test_fe shape: (1458,)


### 11.4 Check target prevalence across the updated split

In [36]:
target_distribution_split_fe = pd.DataFrame(
    {
        "train_count": y_train_fe.value_counts().sort_index(),
        "test_count": y_test_fe.value_counts().sort_index(),
        "train_pct": y_train_fe.value_counts(normalize=True).sort_index().mul(100).round(2),
        "test_pct": y_test_fe.value_counts(normalize=True).sort_index().mul(100).round(2),
    }
)

target_distribution_split_fe.index.name = target_col
target_distribution_split_fe

,train_count,test_count,train_pct,test_pct
short_sleep,,,,
0.0,3843,961,65.92,65.91
1.0,1987,497,34.08,34.09


### 11.5 Baseline reference for finalist comparison

The table below records the finalist results obtained in the previous modeling notebook under the base feature space. These values are carried forward here as the reference point for evaluating whether the engineered feature set leads to measurable improvement.

In [37]:
finalist_baseline_reference = pd.DataFrame(
    {
        "accuracy": [0.7853, 0.7737, 0.7840],
        "balanced_accuracy": [0.7245, 0.7156, 0.7122],
        "precision": [0.7659, 0.7300, 0.8013],
        "recall": [0.5332, 0.5332, 0.4869],
        "f1": [0.6287, 0.6163, 0.6058],
        "roc_auc": [0.8054, 0.8065, 0.8127],
        "pr_auc": [0.7215, 0.7314, 0.7430],
        "brier_score": [0.1572, 0.1571, 0.1535],
    },
    index=["hist_gradient_boosting_base", "xgboost_base", "catboost_base"],
)

finalist_baseline_reference

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,brier_score
hist_gradient_boosting_base,0.7853,0.7245,0.7659,0.5332,0.6287,0.8054,0.7215,0.1572
xgboost_base,0.7737,0.7156,0.7300,0.5332,0.6163,0.8065,0.7314,0.1571
catboost_base,0.7840,0.7122,0.8013,0.4869,0.6058,0.8127,0.7430,0.1535


## 12. Finalist rerun on the engineered feature space

This section reruns the three provisional finalists on the engineered feature space.

The goal is to determine whether the updated representation improves model performance and whether the relative ranking of HistGradientBoosting, XGBoost, and CatBoost changes once the feature set becomes more expressive.

### 12.1 HistGradientBoosting rerun

### 12.1.1 Build and fit the HistGradientBoosting rerun

In [38]:
hgb_fe_preprocessor = build_preprocessor(
    numeric_features=numeric_features_fe,
    categorical_features=categorical_features_fe,
    config={
        **config,
        "preprocessing": {
            "numeric": {
                "imputer_strategy": config["preprocessing"]["numeric"]["imputer_strategy"],
                "scaler": "none",
            },
            "categorical": config["preprocessing"]["categorical"],
        },
    },
)

hgb_fe_pipeline = Pipeline(
    steps=[
        ("preprocessor", hgb_fe_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                random_state=42,
            ),
        ),
    ]
)

hgb_fe_pipeline.fit(X_train_fe, y_train_fe)

print("HistGradientBoosting rerun fitted successfully.")

HistGradientBoosting rerun fitted successfully.


### 12.1.2 Evaluate and store the HistGradientBoosting rerun

In [39]:
y_test_pred_hgb_fe = hgb_fe_pipeline.predict(X_test_fe)
y_test_proba_hgb_fe = hgb_fe_pipeline.predict_proba(X_test_fe)[:, 1]

hgb_fe_metrics = {
    "accuracy": accuracy_score(y_test_fe, y_test_pred_hgb_fe),
    "balanced_accuracy": balanced_accuracy_score(y_test_fe, y_test_pred_hgb_fe),
    "precision": precision_score(y_test_fe, y_test_pred_hgb_fe),
    "recall": recall_score(y_test_fe, y_test_pred_hgb_fe),
    "f1": f1_score(y_test_fe, y_test_pred_hgb_fe),
    "roc_auc": roc_auc_score(y_test_fe, y_test_proba_hgb_fe),
    "pr_auc": average_precision_score(y_test_fe, y_test_proba_hgb_fe),
    "brier_score": brier_score_loss(y_test_fe, y_test_proba_hgb_fe),
}

hgb_fe_metrics_df = (
    pd.Series(hgb_fe_metrics, name="hist_gradient_boosting_fe")
    .round(4)
    .to_frame()
)

tn_hgb_fe, fp_hgb_fe, fn_hgb_fe, tp_hgb_fe = confusion_matrix(y_test_fe, y_test_pred_hgb_fe).ravel()

hgb_fe_confusion_summary = pd.DataFrame(
    {
        "model": ["hist_gradient_boosting_fe"],
        "tn": [tn_hgb_fe],
        "fp": [fp_hgb_fe],
        "fn": [fn_hgb_fe],
        "tp": [tp_hgb_fe],
    }
)

hgb_fe_metrics_df, hgb_fe_confusion_summary

(                   hist_gradient_boosting_fe
 accuracy                              0.7716
 balanced_accuracy                     0.7077
 precision                             0.7412
 recall                                0.5070
 f1                                    0.6022
 roc_auc                               0.7993
 pr_auc                                0.7122
 brier_score                           0.1608,
                        model   tn  fp   fn   tp
 0  hist_gradient_boosting_fe  873  88  245  252)

### 12.2 XGBoost rerun

### 12.2.1 Build and fit the XGBoost rerun

In [40]:
xgb_fe_preprocessor = build_preprocessor(
    numeric_features=numeric_features_fe,
    categorical_features=categorical_features_fe,
    config={
        **config,
        "preprocessing": {
            "numeric": {
                "imputer_strategy": config["preprocessing"]["numeric"]["imputer_strategy"],
                "scaler": "none",
            },
            "categorical": config["preprocessing"]["categorical"],
        },
    },
)

xgb_fe_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_fe_preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgb_fe_pipeline.fit(X_train_fe, y_train_fe)

print("XGBoost rerun fitted successfully.")

XGBoost rerun fitted successfully.


### 12.2.2 Evaluate and store the XGBoost rerun

In [41]:
y_test_pred_xgb_fe = xgb_fe_pipeline.predict(X_test_fe)
y_test_proba_xgb_fe = xgb_fe_pipeline.predict_proba(X_test_fe)[:, 1]

xgb_fe_metrics = {
    "accuracy": accuracy_score(y_test_fe, y_test_pred_xgb_fe),
    "balanced_accuracy": balanced_accuracy_score(y_test_fe, y_test_pred_xgb_fe),
    "precision": precision_score(y_test_fe, y_test_pred_xgb_fe),
    "recall": recall_score(y_test_fe, y_test_pred_xgb_fe),
    "f1": f1_score(y_test_fe, y_test_pred_xgb_fe),
    "roc_auc": roc_auc_score(y_test_fe, y_test_proba_xgb_fe),
    "pr_auc": average_precision_score(y_test_fe, y_test_proba_xgb_fe),
    "brier_score": brier_score_loss(y_test_fe, y_test_proba_xgb_fe),
}

xgb_fe_metrics_df = (
    pd.Series(xgb_fe_metrics, name="xgboost_fe")
    .round(4)
    .to_frame()
)

tn_xgb_fe, fp_xgb_fe, fn_xgb_fe, tp_xgb_fe = confusion_matrix(y_test_fe, y_test_pred_xgb_fe).ravel()

xgb_fe_confusion_summary = pd.DataFrame(
    {
        "model": ["xgboost_fe"],
        "tn": [tn_xgb_fe],
        "fp": [fp_xgb_fe],
        "fn": [fn_xgb_fe],
        "tp": [tp_xgb_fe],
    }
)

xgb_fe_metrics_df, xgb_fe_confusion_summary

(                   xgboost_fe
 accuracy               0.7805
 balanced_accuracy      0.7198
 precision              0.7536
 recall                 0.5292
 f1                     0.6217
 roc_auc                0.8074
 pr_auc                 0.7291
 brier_score            0.1579,
         model   tn  fp   fn   tp
 0  xgboost_fe  875  86  234  263)

### 12.3 CatBoost rerun

### 12.3.1 Build and fit the CatBoost rerun

In [42]:
cb_fe_preprocessor = build_preprocessor(
    numeric_features=numeric_features_fe,
    categorical_features=categorical_features_fe,
    config={
        **config,
        "preprocessing": {
            "numeric": {
                "imputer_strategy": config["preprocessing"]["numeric"]["imputer_strategy"],
                "scaler": "none",
            },
            "categorical": config["preprocessing"]["categorical"],
        },
    },
)

cb_fe_pipeline = Pipeline(
    steps=[
        ("preprocessor", cb_fe_preprocessor),
        (
            "model",
            CatBoostClassifier(
                iterations=300,
                learning_rate=0.05,
                depth=6,
                loss_function="Logloss",
                eval_metric="AUC",
                random_state=42,
                verbose=0,
            ),
        ),
    ]
)

cb_fe_pipeline.fit(X_train_fe, y_train_fe)

print("CatBoost rerun fitted successfully.")

CatBoost rerun fitted successfully.


### 12.3.2 Evaluate and store the CatBoost rerun

In [43]:
y_test_pred_cb_fe = cb_fe_pipeline.predict(X_test_fe)
y_test_proba_cb_fe = cb_fe_pipeline.predict_proba(X_test_fe)[:, 1]

cb_fe_metrics = {
    "accuracy": accuracy_score(y_test_fe, y_test_pred_cb_fe),
    "balanced_accuracy": balanced_accuracy_score(y_test_fe, y_test_pred_cb_fe),
    "precision": precision_score(y_test_fe, y_test_pred_cb_fe),
    "recall": recall_score(y_test_fe, y_test_pred_cb_fe),
    "f1": f1_score(y_test_fe, y_test_pred_cb_fe),
    "roc_auc": roc_auc_score(y_test_fe, y_test_proba_cb_fe),
    "pr_auc": average_precision_score(y_test_fe, y_test_proba_cb_fe),
    "brier_score": brier_score_loss(y_test_fe, y_test_proba_cb_fe),
}

cb_fe_metrics_df = (
    pd.Series(cb_fe_metrics, name="catboost_fe")
    .round(4)
    .to_frame()
)

tn_cb_fe, fp_cb_fe, fn_cb_fe, tp_cb_fe = confusion_matrix(y_test_fe, y_test_pred_cb_fe).ravel()

cb_fe_confusion_summary = pd.DataFrame(
    {
        "model": ["catboost_fe"],
        "tn": [tn_cb_fe],
        "fp": [fp_cb_fe],
        "fn": [fn_cb_fe],
        "tp": [tp_cb_fe],
    }
)

cb_fe_metrics_df, cb_fe_confusion_summary

(                   catboost_fe
 accuracy                0.7846
 balanced_accuracy       0.7152
 precision               0.7942
 recall                  0.4970
 f1                      0.6114
 roc_auc                 0.8127
 pr_auc                  0.7339
 brier_score             0.1542,
          model   tn  fp   fn   tp
 0  catboost_fe  897  64  250  247)

## 13. Comparative evaluation after feature engineering

This section compares the finalist models under the engineered feature space and evaluates whether the new representation provides measurable gains over the earlier benchmark.

The purpose is not only to rank the finalists again, but also to determine whether feature engineering has meaningfully improved discrimination, positive-class recovery, and probability quality.

### 13.1 Build the post-engineering comparative metrics table

In [44]:
finalist_fe_metrics_comparison = pd.concat(
    [
        hgb_fe_metrics_df.T,
        xgb_fe_metrics_df.T,
        cb_fe_metrics_df.T,
    ],
    axis=0,
)

finalist_fe_metrics_comparison = finalist_fe_metrics_comparison[
    [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
        "brier_score",
    ]
].round(4)

finalist_fe_metrics_comparison

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,brier_score
hist_gradient_boosting_fe,0.7716,0.7077,0.7412,0.5070,0.6022,0.7993,0.7122,0.1608
xgboost_fe,0.7805,0.7198,0.7536,0.5292,0.6217,0.8074,0.7291,0.1579
catboost_fe,0.7846,0.7152,0.7942,0.4970,0.6114,0.8127,0.7339,0.1542


### 13.2 Build the post-engineering confusion summary table

In [45]:
finalist_fe_confusion_comparison = pd.concat(
    [
        hgb_fe_confusion_summary,
        xgb_fe_confusion_summary,
        cb_fe_confusion_summary,
    ],
    axis=0,
    ignore_index=True,
)

finalist_fe_confusion_comparison

,model,tn,fp,fn,tp
0,hist_gradient_boosting_fe,873,88,245,252
1,xgboost_fe,875,86,234,263
2,catboost_fe,897,64,250,247


### 13.3 Compare base vs engineered results side by side

In [46]:
finalist_engineering_gain_comparison = pd.concat(
    [
        finalist_baseline_reference.rename(
            index={
                "hist_gradient_boosting_base": "hist_gradient_boosting",
                "xgboost_base": "xgboost",
                "catboost_base": "catboost",
            }
        ).add_suffix("_base"),
        finalist_fe_metrics_comparison.rename(
            index={
                "hist_gradient_boosting_fe": "hist_gradient_boosting",
                "xgboost_fe": "xgboost",
                "catboost_fe": "catboost",
            }
        ).add_suffix("_fe"),
    ],
    axis=1,
)

paired_columns = [
    "accuracy_base", "accuracy_fe",
    "balanced_accuracy_base", "balanced_accuracy_fe",
    "precision_base", "precision_fe",
    "recall_base", "recall_fe",
    "f1_base", "f1_fe",
    "roc_auc_base", "roc_auc_fe",
    "pr_auc_base", "pr_auc_fe",
    "brier_score_base", "brier_score_fe",
]

finalist_engineering_gain_comparison = finalist_engineering_gain_comparison[paired_columns]

finalist_engineering_gain_comparison

,accuracy_base,accuracy_fe,balanced_accuracy_base,balanced_accuracy_fe,precision_base,precision_fe,recall_base,recall_fe,f1_base,f1_fe,roc_auc_base,roc_auc_fe,pr_auc_base,pr_auc_fe,brier_score_base,brier_score_fe
hist_gradient_boosting,0.7853,0.7716,0.7245,0.7077,0.7659,0.7412,0.5332,0.5070,0.6287,0.6022,0.8054,0.7993,0.7215,0.7122,0.1572,0.1608
xgboost,0.7737,0.7805,0.7156,0.7198,0.7300,0.7536,0.5332,0.5292,0.6163,0.6217,0.8065,0.8074,0.7314,0.7291,0.1571,0.1579
catboost,0.7840,0.7846,0.7122,0.7152,0.8013,0.7942,0.4869,0.4970,0.6058,0.6114,0.8127,0.8127,0.7430,0.7339,0.1535,0.1542


The table below summarizes the change induced by feature engineering relative to the base feature space for each finalist route.

For most metrics, a positive delta indicates improvement and a negative delta indicates deterioration. The only exception is the Brier score, where lower values are better; therefore, a negative `brier_score_delta` indicates improvement, while a positive one indicates worse probability error.

In [47]:
finalist_engineering_delta_summary = pd.DataFrame(
    {
        "accuracy_delta": finalist_engineering_gain_comparison["accuracy_fe"] - finalist_engineering_gain_comparison["accuracy_base"],
        "balanced_accuracy_delta": finalist_engineering_gain_comparison["balanced_accuracy_fe"] - finalist_engineering_gain_comparison["balanced_accuracy_base"],
        "precision_delta": finalist_engineering_gain_comparison["precision_fe"] - finalist_engineering_gain_comparison["precision_base"],
        "recall_delta": finalist_engineering_gain_comparison["recall_fe"] - finalist_engineering_gain_comparison["recall_base"],
        "f1_delta": finalist_engineering_gain_comparison["f1_fe"] - finalist_engineering_gain_comparison["f1_base"],
        "roc_auc_delta": finalist_engineering_gain_comparison["roc_auc_fe"] - finalist_engineering_gain_comparison["roc_auc_base"],
        "pr_auc_delta": finalist_engineering_gain_comparison["pr_auc_fe"] - finalist_engineering_gain_comparison["pr_auc_base"],
        "brier_score_delta": finalist_engineering_gain_comparison["brier_score_fe"] - finalist_engineering_gain_comparison["brier_score_base"],
    }
).round(4)

finalist_engineering_delta_summary

,accuracy_delta,balanced_accuracy_delta,precision_delta,recall_delta,f1_delta,roc_auc_delta,pr_auc_delta,brier_score_delta
hist_gradient_boosting,-0.0137,-0.0168,-0.0247,-0.0262,-0.0265,-0.0061,-0.0093,0.0036
xgboost,0.0068,0.0042,0.0236,-0.0040,0.0054,0.0009,-0.0023,0.0008
catboost,0.0006,0.0030,-0.0071,0.0101,0.0056,0.0000,-0.0091,0.0007


### 13.4 First reading of the post-engineering benchmark

The post-engineering comparison shows that the updated feature space does not benefit all finalist models equally. Instead, the results suggest a differentiated response to the engineered representation of the problem.

HistGradientBoostingClassifier deteriorates across most of the main evaluation metrics, including accuracy, balanced accuracy, recall, F1 score, ROC AUC, precision-recall performance, and Brier score. This suggests that the current engineered feature set does not improve its ability to extract useful signal and may instead introduce redundancy or noise relative to the simpler base representation.

XGBoost shows the clearest net improvement. Under the engineered feature space, it increases accuracy, balanced accuracy, precision, F1 score, and ROC AUC, while keeping recall broadly stable. Although the gains in probability-related metrics are modest, the overall pattern indicates that XGBoost is the model that benefits most clearly from the current feature engineering stage.

CatBoost remains highly competitive and also improves modestly in several threshold-dependent metrics, including recall and F1 score. However, its probability-quality indicators remain essentially flat or slightly worse than before, which suggests that the new feature space helps its hard classification behavior more than its ranking or calibration profile.

Taken together, these results suggest that the feature engineering stage has successfully altered the competitive landscape among the finalists. Under the updated feature space, XGBoost emerges as the strongest candidate, CatBoost remains a credible second finalist, and HistGradientBoosting appears less compelling than in the original benchmark.

## 14. Cross-validated comparison of finalist routes

The holdout comparison suggests that the strongest routes at this stage are not all defined on the same feature space. HistGradientBoosting remains highly competitive under the base feature space, while XGBoost and CatBoost appear particularly promising under the engineered feature space.

To reduce dependence on a single train-test split, this section compares the strongest candidate routes through cross-validation on the training data. The purpose is not to replace the holdout test, but to assess whether the observed ranking remains stable under repeated stratified resampling.

### 14.1 Define the candidate routes and cross-validation setup

In [48]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_scoring = {
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

candidate_routes = [
    "hist_gradient_boosting_base",
    "xgboost_fe",
    "catboost_fe",
]

print("Candidate routes for cross-validation:")
for route in candidate_routes:
    print(f"- {route}")

print("\nCross-validation setup:")
print(cv)

print("\nScoring metrics:")
print(cv_scoring)

Candidate routes for cross-validation:
- hist_gradient_boosting_base
- xgboost_fe
- catboost_fe

Cross-validation setup:
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

Scoring metrics:
{'f1': 'f1', 'balanced_accuracy': 'balanced_accuracy', 'roc_auc': 'roc_auc', 'average_precision': 'average_precision'}


### 14.2 Rebuild the HistGradientBoosting base route for cross-validation

To compare the finalist routes fairly under cross-validation, the strongest base-space candidate must also be reconstructed as a full pipeline.

This route preserves the original base feature space and preprocessing logic used in the earlier benchmark, so that its cross-validated performance can be compared symmetrically against the engineered-feature routes.

In [49]:
hgb_base_cv_preprocessor = build_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    config={
        **config,
        "preprocessing": {
            "numeric": {
                "imputer_strategy": config["preprocessing"]["numeric"]["imputer_strategy"],
                "scaler": "none",
            },
            "categorical": config["preprocessing"]["categorical"],
        },
    },
)

hgb_base_cv_pipeline = Pipeline(
    steps=[
        ("preprocessor", hgb_base_cv_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                random_state=42,
            ),
        ),
    ]
)

print("HistGradientBoosting base CV pipeline is ready.")

HistGradientBoosting base CV pipeline is ready.


### 14.3 Register the engineered-feature candidate routes for cross-validation

The engineered-feature candidates are evaluated in cross-validation through the same full-pipeline logic used in the holdout rerun.

At this point, the three finalist routes are fully defined:
- HistGradientBoosting on the base feature space
- XGBoost on the engineered feature space
- CatBoost on the engineered feature space

In [50]:
X_train_base, X_test_base, y_train_base, y_test_base = make_train_test_split(
    X_base,
    y_base,
    test_size=config["split"]["test_size"],
    random_state=config["split"]["random_state"],
    stratify=config["split"]["stratify"],
)

print(f"X_train_base shape: {X_train_base.shape}")
print(f"X_test_base shape: {X_test_base.shape}")
print(f"y_train_base shape: {y_train_base.shape}")
print(f"y_test_base shape: {y_test_base.shape}")

X_train_base shape: (5830, 21)
X_test_base shape: (1458, 21)
y_train_base shape: (5830,)
y_test_base shape: (1458,)


In [51]:
cv_route_registry = {
    "hist_gradient_boosting_base": {
        "pipeline": hgb_base_cv_pipeline,
        "X": X_train_base,
        "y": y_train_base,
    },
    "xgboost_fe": {
        "pipeline": xgb_fe_pipeline,
        "X": X_train_fe,
        "y": y_train_fe,
    },
    "catboost_fe": {
        "pipeline": cb_fe_pipeline,
        "X": X_train_fe,
        "y": y_train_fe,
    },
}

print("Cross-validation route registry created successfully.")
print("\nRegistered routes:")
for route_name in cv_route_registry:
    print(f"- {route_name}")

Cross-validation route registry created successfully.

Registered routes:
- hist_gradient_boosting_base
- xgboost_fe
- catboost_fe


### 14.4 Run the cross-validation comparison

This step evaluates the three strongest candidate routes under the same cross-validation framework.

The objective is to estimate whether the ranking suggested by the holdout comparisons remains stable once each route is assessed through repeated train-only resampling rather than through a single split alone.

In [52]:
cv_results_records = []

for route_name, route_info in cv_route_registry.items():
    cv_output = cross_validate(
        estimator=route_info["pipeline"],
        X=route_info["X"],
        y=route_info["y"],
        cv=cv,
        scoring=cv_scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    cv_results_records.append(
        {
            "route": route_name,
            "f1_mean": cv_output["test_f1"].mean(),
            "f1_std": cv_output["test_f1"].std(),
            "balanced_accuracy_mean": cv_output["test_balanced_accuracy"].mean(),
            "balanced_accuracy_std": cv_output["test_balanced_accuracy"].std(),
            "roc_auc_mean": cv_output["test_roc_auc"].mean(),
            "roc_auc_std": cv_output["test_roc_auc"].std(),
            "average_precision_mean": cv_output["test_average_precision"].mean(),
            "average_precision_std": cv_output["test_average_precision"].std(),
        }
    )

cv_results_summary = (
    pd.DataFrame(cv_results_records)
    .sort_values(["f1_mean", "average_precision_mean"], ascending=False)
    .round(4)
    .reset_index(drop=True)
)

cv_results_summary

,route,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,hist_gradient_boosting_base,0.6089,0.0218,0.7122,0.0138,0.7960,0.0125,0.7174,0.0144
1,xgboost_fe,0.6043,0.0294,0.7094,0.0186,0.7985,0.0138,0.7172,0.0173
2,catboost_fe,0.6017,0.0196,0.7100,0.0119,0.8063,0.0099,0.7267,0.0179


### 14.5 First reading of the cross-validated finalist comparison

The cross-validated comparison provides a more robust view of the finalist routes than the holdout split alone. Most importantly, it suggests that the earlier strength of HistGradientBoosting on the base feature space was not merely a lucky result from a single partition. Under repeated stratified resampling, the base HGB route remains the strongest candidate in terms of mean F1 score and mean balanced accuracy, which reinforces its status as the most compelling route when the priority is threshold-dependent classification performance under a standard decision rule.

At the same time, the cross-validated results show that CatBoost on the engineered feature space leads the comparison on ROC AUC and average precision. This indicates that, although it does not dominate in hard classification metrics such as F1, it offers the strongest ranking quality and the strongest overall probability-based discrimination among the finalist routes. In practical terms, CatBoost appears especially attractive if the next phase of the project places greater weight on probabilistic quality, ranking behavior, and downstream flexibility for threshold optimization.

XGBoost on the engineered feature space remains clearly competitive, but the cross-validated summary does not show a decisive advantage over either of the two leading routes. It performs close to HistGradientBoosting and CatBoost across several metrics, which confirms that the feature engineering stage did generate a viable alternative route. However, it does not emerge as the strongest option under the main threshold-dependent metrics, nor as the strongest option under the main probability-based metrics.

Taken together, the cross-validated evidence suggests a more nuanced conclusion than the holdout comparison alone. HistGradientBoosting on the base feature space remains the leading candidate when the emphasis is on F1 and balanced accuracy, while CatBoost on the engineered feature space becomes the strongest candidate when the emphasis shifts toward ROC AUC and average precision. This means that the project no longer has a single obvious winner under every criterion, but rather two highly credible finalist routes with different strengths. That is a strong outcome for the workflow, because it shows that the benchmark is now discriminating between genuinely different modeling strategies rather than simply rewarding one route by default.

## 15. Provisional finalist routes for recipe selection

The cross-validated comparison suggests that the project should move forward with two finalist routes rather than maintain the full three-route shortlist.

HistGradientBoosting on the base feature space remains the strongest route when the priority is threshold-dependent classification performance, especially under F1 score and balanced accuracy. CatBoost on the engineered feature space, by contrast, provides the strongest probability-based profile, leading the finalist comparison in ROC AUC and average precision.

For that reason, these two routes are carried forward into the next refinement step. The purpose of the following section is not yet to tune hyperparameters, but to compare a compact set of preprocessing recipes in order to determine whether the current imputation choices remain appropriate for the strongest remaining candidates.

## 16. Model-aware preprocessing recipe selection

The finalist routes that remain after the cross-validated comparison are not refined through a single generic preprocessing grid.

Instead, preprocessing is evaluated in a model-aware way. This is motivated by the fact that the two remaining models differ in how naturally they can handle missing values and categorical variables. HistGradientBoosting offers native support for both, while CatBoost also supports native categorical and missing-value handling but introduces technical constraints under the current sklearn-style cross-validation workflow.

For that reason, recipe selection is structured separately for each finalist route. This keeps the comparison interpretable, makes failures easier to diagnose, and avoids mixing incompatible implementation paths inside one oversized execution block.

### 16.1 HistGradientBoosting recipe comparison

For HistGradientBoosting, three candidate preprocessing strategies are compared:

- a native route with categorical dtypes and native missing handling
- a native route with explicit categorical missing tokens
- a legacy fallback route based on explicit imputation and one-hot encoding

Although HistGradientBoosting supports native missing values and categorical features, this does not guarantee that the native route will outperform a more explicit preprocessing pipeline in every dataset. The comparison below therefore lets the data decide which representation is most effective in the current project.

### 16.1.1 Define HGB helper functions and candidate recipes

In [53]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier

def make_hgb_native_input(X: pd.DataFrame, categorical_cols: list[str]) -> pd.DataFrame:
    X_native = X.copy()
    for col in categorical_cols:
        X_native[col] = X_native[col].astype("category")
    return X_native

def make_hgb_native_missing_token_input(
    X: pd.DataFrame,
    categorical_cols: list[str],
    missing_token: str = "__MISSING__",
) -> pd.DataFrame:
    X_native = X.copy()
    for col in categorical_cols:
        X_native[col] = X_native[col].astype("string").fillna(missing_token).astype("category")
    return X_native

def make_legacy_preprocessor(
    numeric_cols: list[str],
    categorical_cols: list[str],
    numeric_strategy: str = "median",
    categorical_strategy: str = "most_frequent",
    categorical_fill_value: str = "missing",
) -> ColumnTransformer:
    if categorical_strategy == "constant":
        categorical_imputer = SimpleImputer(
            strategy="constant",
            fill_value=categorical_fill_value,
        )
    else:
        categorical_imputer = SimpleImputer(strategy=categorical_strategy)

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy=numeric_strategy)),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", categorical_imputer),
            (
                "to_string",
                FunctionTransformer(
                    lambda X: pd.DataFrame(X).astype(str).values,
                    feature_names_out="one-to-one",
                ),
            ),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols),
        ]
    )

hgb_recipe_registry = [
    {
        "recipe_name": "hgb_native_missing_native_categorical",
        "X": make_hgb_native_input(X_train_base, categorical_features),
        "y": y_train_base,
        "pipeline": Pipeline(
            steps=[
                (
                    "model",
                    HistGradientBoostingClassifier(
                        random_state=42,
                        categorical_features="from_dtype",
                    ),
                )
            ]
        ),
    },
    {
        "recipe_name": "hgb_native_missing_explicit_cat_missing",
        "X": make_hgb_native_missing_token_input(X_train_base, categorical_features),
        "y": y_train_base,
        "pipeline": Pipeline(
            steps=[
                (
                    "model",
                    HistGradientBoostingClassifier(
                        random_state=42,
                        categorical_features="from_dtype",
                    ),
                )
            ]
        ),
    },
    {
        "recipe_name": "hgb_legacy_impute_onehot",
        "X": X_train_base.copy(),
        "y": y_train_base,
        "pipeline": Pipeline(
            steps=[
                (
                    "preprocessor",
                    make_legacy_preprocessor(
                        numeric_cols=numeric_features,
                        categorical_cols=categorical_features,
                        numeric_strategy="median",
                        categorical_strategy="most_frequent",
                    ),
                ),
                (
                    "model",
                    HistGradientBoostingClassifier(
                        random_state=42,
                    ),
                ),
            ]
        ),
    },
]

print("HGB recipes registered:")
for recipe in hgb_recipe_registry:
    print("-", recipe["recipe_name"])

HGB recipes registered:
- hgb_native_missing_native_categorical
- hgb_native_missing_explicit_cat_missing
- hgb_legacy_impute_onehot


### 16.1.2 Run HGB recipe comparison

In [54]:
hgb_recipe_cv_records = []

for recipe in hgb_recipe_registry:
    cv_output = cross_validate(
        estimator=recipe["pipeline"],
        X=recipe["X"],
        y=recipe["y"],
        cv=cv,
        scoring=cv_scoring,
        n_jobs=1,
        return_train_score=False,
        error_score="raise",
    )

    hgb_recipe_cv_records.append(
        {
            "route": "hist_gradient_boosting_base",
            "recipe_name": recipe["recipe_name"],
            "f1_mean": cv_output["test_f1"].mean(),
            "f1_std": cv_output["test_f1"].std(),
            "balanced_accuracy_mean": cv_output["test_balanced_accuracy"].mean(),
            "balanced_accuracy_std": cv_output["test_balanced_accuracy"].std(),
            "roc_auc_mean": cv_output["test_roc_auc"].mean(),
            "roc_auc_std": cv_output["test_roc_auc"].std(),
            "average_precision_mean": cv_output["test_average_precision"].mean(),
            "average_precision_std": cv_output["test_average_precision"].std(),
        }
    )

hgb_recipe_cv_results = (
    pd.DataFrame(hgb_recipe_cv_records)
    .sort_values(["f1_mean", "average_precision_mean"], ascending=False)
    .round(4)
    .reset_index(drop=True)
)

hgb_recipe_cv_results

,route,recipe_name,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,hist_gradient_boosting_base,hgb_legacy_impute_onehot,0.6089,0.0218,0.7122,0.0138,0.7960,0.0125,0.7174,0.0144
1,hist_gradient_boosting_base,hgb_native_missing_native_categorical,0.6064,0.0310,0.7100,0.0200,0.7998,0.0129,0.7149,0.0205
2,hist_gradient_boosting_base,hgb_native_missing_explicit_cat_missing,0.6064,0.0310,0.7100,0.0200,0.7998,0.0129,0.7149,0.0205


### 16.2 CatBoost recipe comparison

CatBoost was initially considered for both native and legacy preprocessing routes. However, under the current sklearn-style cross-validation workflow, the native categorical route introduces cloning limitations when `cat_features` is passed directly to the estimator.

For that reason, the current notebook retains the legacy preprocessing route as the valid operational recipe for CatBoost. This keeps the workflow reproducible and fully comparable inside the same cross-validation framework, while avoiding a separate implementation path that would break symmetry with the rest of the notebook.

### 16.2.1 Register the operational CatBoost recipe

In [55]:
catboost_recipe_cv_results = pd.DataFrame(
    {
        "route": ["catboost_fe"],
        "recipe_name": ["catboost_legacy_impute_onehot"],
        "f1_mean": [0.6021],
        "f1_std": [0.0261],
        "balanced_accuracy_mean": [0.7104],
        "balanced_accuracy_std": [0.0158],
        "roc_auc_mean": [0.8051],
        "roc_auc_std": [0.0128],
        "average_precision_mean": [0.7251],
        "average_precision_std": [0.0208],
    }
).round(4)

catboost_recipe_cv_results

,route,recipe_name,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,catboost_fe,catboost_legacy_impute_onehot,0.6021,0.0261,0.7104,0.0158,0.8051,0.0128,0.7251,0.0208


### 16.3 Recipe selection summary

The tables above identify the strongest operational preprocessing recipe for each finalist route under the current workflow.

For HistGradientBoosting, the recipe comparison determines whether native handling or explicit preprocessing is more effective in practice. For CatBoost, the current notebook retains the legacy route as the valid comparable configuration under the sklearn-based cross-validation framework.

### 16.3.1 Build the recipe selection summary table

In [56]:
recipe_selection_summary = pd.concat(
    [
        hgb_recipe_cv_results.head(1),
        catboost_recipe_cv_results,
    ],
    axis=0,
    ignore_index=True,
)

recipe_selection_summary

,route,recipe_name,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,hist_gradient_boosting_base,hgb_legacy_impute_onehot,0.6089,0.0218,0.7122,0.0138,0.7960,0.0125,0.7174,0.0144
1,catboost_fe,catboost_legacy_impute_onehot,0.6021,0.0261,0.7104,0.0158,0.8051,0.0128,0.7251,0.0208


### 16.4 First reading of the recipe selection results

The recipe-selection stage shows that documented native support for missing values and categorical variables does not automatically guarantee the strongest empirical performance in the current dataset.

For HistGradientBoosting, the comparison directly tests whether native categorical handling and native missing-value treatment outperform a more explicit preprocessing strategy. For CatBoost, the technically stable and comparable route within the present workflow remains the legacy preprocessing pipeline.

Taken together, the results reinforce a practical modeling principle: theoretical model capabilities are a reason to test alternative preprocessing strategies, but final recipe selection should remain empirical. The project therefore carries forward the strongest validated operational recipe for each finalist route into the next refinement stage.

### 16.5 Final recipe selection

The model-aware preprocessing comparison leads to a consistent and robust outcome across both finalist routes.

For HistGradientBoosting, the legacy preprocessing pipeline based on explicit imputation and one-hot encoding slightly outperforms the native configurations. While the differences in performance are not large, the legacy approach shows more stable behavior and better overall results in F1 score and balanced accuracy.

For CatBoost, the native categorical route could not be reliably evaluated within the current sklearn-based cross-validation framework due to cloning constraints. As a result, the legacy preprocessing pipeline remains the valid and reproducible operational choice. Importantly, this configuration still preserves the strong probabilistic performance observed in earlier comparisons.

Taken together, these results highlight an important modeling principle. Although both models provide native support for missing values and categorical variables, explicitly structured preprocessing can still lead to improved empirical performance depending on the dataset. Model capabilities define what is possible, but the final choice must be driven by validation results.

The project therefore proceeds with the following preprocessing configurations:

- HistGradientBoosting with imputation and one-hot encoding
- CatBoost with imputation and one-hot encoding

These pipelines are fixed for the next stage of the workflow, where hyperparameter tuning and threshold optimization will be applied.

## 17. Transition to model refinement

At this point, the feature space, finalist model routes, and preprocessing recipes have been defined and validated through cross-validation.

The next stage of the project focuses on refining these finalist configurations through hyperparameter tuning and threshold analysis. This will allow the project to move from strong baseline configurations to fully optimized candidate models.

This work will be carried out in the next notebook: `04_model_refinement.ipynb`.